# Appendix G: TensorFlow Graphs

### G1: Exploring Function Definition and Graphs

In [88]:
import tensorflow as tf

@tf.function
def kinetic_energy(m: float, v: float) -> float:
    return m * v ** 2

concrete_func_float = kinetic_energy.get_concrete_function(1, 2)
concrete_func_tensor = kinetic_energy.get_concrete_function(tf.constant([1, 2], dtype=tf.float32), tf.constant([1, 2], dtype=tf.float32))

print(concrete_func_float)
print(concrete_func_tensor)

ConcreteFunction kinetic_energy(m=1, v=2)
  Returns:
    int32 Tensor, shape=()
ConcreteFunction kinetic_energy(m, v)
  Args:
    m: float32 Tensor, shape=(2,)
    v: float32 Tensor, shape=(2,)
  Returns:
    float32 Tensor, shape=(2,)


In [89]:
concrete_func_float.graph

In [90]:
concrete_func_float.graph.get_operations()

[<tf.Operation 'Const' type=Const>, <tf.Operation 'Identity' type=Identity>]

In [91]:
concrete_func_tensor.graph

In [92]:
ops = concrete_func_tensor.graph.get_operations()
ops

[<tf.Operation 'm' type=Placeholder>,
 <tf.Operation 'v' type=Placeholder>,
 <tf.Operation 'pow/y' type=Const>,
 <tf.Operation 'pow' type=Pow>,
 <tf.Operation 'mul' type=Mul>,
 <tf.Operation 'Identity' type=Identity>]

In [93]:
print(ops[2].inputs)
print(ops[2].outputs)

()
[<tf.Tensor 'pow/y:0' shape=() dtype=float32>]


In [94]:
print(ops[3].inputs)
print(ops[3].outputs)

(<tf.Tensor 'v:0' shape=(2,) dtype=float32>, <tf.Tensor 'pow/y:0' shape=() dtype=float32>)
[<tf.Tensor 'pow:0' shape=(2,) dtype=float32>]


In [95]:
print(concrete_func_tensor.graph.get_tensor_by_name("pow/y:0"))

Tensor("pow/y:0", shape=(), dtype=float32)


In [96]:
print(concrete_func_tensor.graph.get_operation_by_name("pow/y"))

name: "pow/y"
op: "Const"
attr {
  key: "dtype"
  value {
    type: DT_FLOAT
  }
}
attr {
  key: "value"
  value {
    tensor {
      dtype: DT_FLOAT
      tensor_shape {
      }
      float_val: 2.0
    }
  }
}



In [97]:
print(concrete_func_tensor.graph.get_operation_by_name("pow"))

name: "pow"
op: "Pow"
input: "v"
input: "pow/y"
attr {
  key: "T"
  value {
    type: DT_FLOAT
  }
}



In [98]:
print(concrete_func_tensor.function_def)

signature {
  name: "__inference_kinetic_energy_878"
  input_arg {
    name: "m"
    type: DT_FLOAT
  }
  input_arg {
    name: "v"
    type: DT_FLOAT
  }
  output_arg {
    name: "identity"
    type: DT_FLOAT
  }
}
node_def {
  name: "pow/y"
  op: "Const"
  attr {
    key: "dtype"
    value {
      type: DT_FLOAT
    }
  }
  attr {
    key: "value"
    value {
      tensor {
        dtype: DT_FLOAT
        tensor_shape {
        }
        float_val: 2.0
      }
    }
  }
  experimental_debug_info {
    original_node_names: "pow/y"
  }
}
node_def {
  name: "pow"
  op: "Pow"
  input: "v"
  input: "pow/y:output:0"
  attr {
    key: "T"
    value {
      type: DT_FLOAT
    }
  }
  experimental_debug_info {
    original_node_names: "pow"
  }
}
node_def {
  name: "mul"
  op: "Mul"
  input: "m"
  input: "pow:z:0"
  attr {
    key: "T"
    value {
      type: DT_FLOAT
    }
  }
  experimental_debug_info {
    original_node_names: "mul"
  }
}
node_def {
  name: "Identity"
  op: "Identity"
  in

### G2: A Closer Look at Tracing

In [99]:
@tf.function
def tf_cube(x: float) -> float:
    print("x = ", x)
    return x ** 3

x = tf.constant(2.)

tf_cube(x)

x =  Tensor("x:0", shape=(), dtype=float32)


<tf.Tensor: shape=(), dtype=float32, numpy=8.0>

In [100]:
x = tf.constant(3.)
tf_cube(x)

<tf.Tensor: shape=(), dtype=float32, numpy=27.0>

In [101]:
x = tf.constant([[3., 4], [5, 6]], dtype=tf.float32)
tf_cube(x)

x =  Tensor("x:0", shape=(2, 2), dtype=float32)


<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[ 27.,  64.],
       [125., 216.]], dtype=float32)>

In [102]:
@tf.function(input_signature=[tf.TensorSpec([None, 2], tf.float32)])
def tf_cube(x: tf.Tensor) -> tf.Tensor:
    return x ** 3

In [103]:
tf_cube(tf.constant([[1, 2], [2, 3], [3, 4]], dtype=tf.float32))

<tf.Tensor: shape=(3, 2), dtype=float32, numpy=
array([[ 1.,  8.],
       [ 8., 27.],
       [27., 64.]], dtype=float32)>

### G3: Using AutoGraph to Capture Control Flow

In [104]:
@tf.function
def add_y(x: tf.Tensor) -> tf.Tensor:
    for i in range(10):
        x += 1
    return x

concrete_fun = add_y.get_concrete_function(tf.constant(0))
concrete_fun.graph.get_operations()

[<tf.Operation 'x' type=Placeholder>,
 <tf.Operation 'add/y' type=Const>,
 <tf.Operation 'add' type=AddV2>,
 <tf.Operation 'add_1/y' type=Const>,
 <tf.Operation 'add_1' type=AddV2>,
 <tf.Operation 'add_2/y' type=Const>,
 <tf.Operation 'add_2' type=AddV2>,
 <tf.Operation 'add_3/y' type=Const>,
 <tf.Operation 'add_3' type=AddV2>,
 <tf.Operation 'add_4/y' type=Const>,
 <tf.Operation 'add_4' type=AddV2>,
 <tf.Operation 'add_5/y' type=Const>,
 <tf.Operation 'add_5' type=AddV2>,
 <tf.Operation 'add_6/y' type=Const>,
 <tf.Operation 'add_6' type=AddV2>,
 <tf.Operation 'add_7/y' type=Const>,
 <tf.Operation 'add_7' type=AddV2>,
 <tf.Operation 'add_8/y' type=Const>,
 <tf.Operation 'add_8' type=AddV2>,
 <tf.Operation 'add_9/y' type=Const>,
 <tf.Operation 'add_9' type=AddV2>,
 <tf.Operation 'Identity' type=Identity>]

In [105]:
@tf.function
def add_y(x: tf.Tensor, y: tf.Tensor) -> tf.Tensor:
    for i in range(y):
        x += 1
    return x

In [106]:
add_y(tf.constant(0))

TypeError: in user code:


    TypeError: tf__add_y() missing 1 required positional argument: 'y'


In [ ]:
concrete_fun = add_y.get_concrete_function(tf.constant(0), tf.constant(1))
concrete_fun.graph.get_operations()

In [ ]:
@tf.function
def add_y(x: tf.Tensor) -> tf.Tensor:
    for i in tf.range(10):
        x += 1
    return x

concrete_fun = add_y.get_concrete_function(tf.constant(0))
concrete_fun.graph.get_operations()

### G4: Handling Variables and Other Resources in TF Functions

In [113]:
counter = tf.Variable(100)
counter
c = tf.Variable(2)
c

<tf.Variable 'Variable:0' shape=() dtype=int32, numpy=2>

In [114]:
@tf.function
def increment(x: tf.Variable, c: tf.int16 = 1) -> tf.Tensor:
    return x.assign_add(c)

In [116]:
increment(counter, c)
increment(counter, c)

<tf.Tensor: shape=(), dtype=int32, numpy=108>

In [118]:
counter

<tf.Variable 'Variable:0' shape=() dtype=int32, numpy=108>

In [120]:
concrete_fun = increment.get_concrete_function(counter, c)
concrete_fun.function_def

signature {
  name: "__inference_increment_1300"
  input_arg {
    name: "x"
    type: DT_RESOURCE
  }
  input_arg {
    name: "c"
    type: DT_RESOURCE
  }
  output_arg {
    name: "identity"
    type: DT_INT32
  }
  is_stateful: true
  control_output: "AssignAddVariableOp"
  control_output: "ReadVariableOp"
  control_output: "ReadVariableOp_1"
}
node_def {
  name: "ReadVariableOp"
  op: "ReadVariableOp"
  input: "c"
  attr {
    key: "dtype"
    value {
      type: DT_INT32
    }
  }
  experimental_debug_info {
    original_node_names: "ReadVariableOp"
  }
}
node_def {
  name: "AssignAddVariableOp"
  op: "AssignAddVariableOp"
  input: "x"
  input: "ReadVariableOp:value:0"
  attr {
    key: "dtype"
    value {
      type: DT_INT32
    }
  }
  experimental_debug_info {
    original_node_names: "AssignAddVariableOp"
  }
}
node_def {
  name: "ReadVariableOp_1"
  op: "ReadVariableOp"
  input: "x"
  input: "^AssignAddVariableOp"
  attr {
    key: "dtype"
    value {
      type: DT_INT32
  

In [122]:
concrete_fun.function_def.signature.input_arg

[name: "x"
type: DT_RESOURCE
, name: "c"
type: DT_RESOURCE
]

### G5: Using TF Functions with `tf.keras` (or not)

### Test Zone

In [ ]:
from collections import OrderedDict
from typing import Dict

print(isinstance({"a": 1}, dict))
print(isinstance(OrderedDict, dict))
print(isinstance({"a": 1}, Dict[str, int]))